# Ixly AssessmentPlatform API Explorer
Haalt de output van alle endpoints op en toont ze als DataFrames. Sla alles op als CSV in de map `csv_output/`.

In [ ]:
# ─── CONFIGURATIE ────────────────────────────────────────────────────────────
BASE_URL      = "https://JOUW_DOMEIN.ixly.nl"   # pas aan
CLIENT_ID     = "JOUW_CLIENT_ID"                 # pas aan
CLIENT_SECRET = "JOUW_CLIENT_SECRET"             # pas aan
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import requests
import pandas as pd
import json
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

CSV_DIR = "csv_output"
os.makedirs(CSV_DIR, exist_ok=True)

# ── OAuth2 token ophalen (Client Credentials flow) ───────────────────────────
def get_token():
    resp = requests.post(
        f"{BASE_URL}/oauth/token",
        data={
            "grant_type":    "client_credentials",
            "client_id":     CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "scope":         "public",
        },
    )
    resp.raise_for_status()
    return resp.json()["access_token"]

TOKEN = get_token()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/json"}
print("Token opgehaald.")


# ── Helpers ───────────────────────────────────────────────────────────────────
def get(path, params=None):
    """GET-verzoek met foutafhandeling. Geeft ruwe JSON terug."""
    resp = requests.get(f"{BASE_URL}{path}", headers=HEADERS, params=params)
    if not resp.ok:
        print(f"  ⚠ {resp.status_code} op {path}: {resp.text[:200]}")
        return None
    return resp.json()


def jsonapi_to_df(data):
    """
    Converteert een JSON:API response naar een platte DataFrame.
    Werkt voor zowel een enkel object als een lijst.
    """
    if data is None:
        return pd.DataFrame()

    items = data.get("data", data)          # sommige endpoints returnen geen envelope
    if isinstance(items, dict):             # enkel object
        items = [items]
    if not isinstance(items, list) or not items:
        return pd.DataFrame()

    rows = []
    for item in items:
        row = {"id": item.get("id"), "type": item.get("type")}
        row.update(item.get("attributes", {}) or {})
        # links platslaan
        for k, v in (item.get("links") or {}).items():
            row[f"link_{k}"] = v
        # relaties: alleen de id's bewaren
        for rel, rel_data in (item.get("relationships") or {}).items():
            inner = (rel_data or {}).get("data")
            if isinstance(inner, dict):
                row[f"rel_{rel}_id"] = inner.get("id")
            elif isinstance(inner, list):
                row[f"rel_{rel}_ids"] = ", ".join(i.get("id", "") for i in inner)
        rows.append(row)

    return pd.DataFrame(rows)


def fetch_paginated(path, params=None):
    """Haalt alle pagina's op en combineert ze."""
    params = dict(params or {})
    params.setdefault("page", 1)
    all_items = []
    while True:
        data = get(path, params)
        if data is None:
            break
        items = data.get("data", [])
        if not items:
            break
        all_items.extend(items)
        next_link = (data.get("links") or {}).get("next")
        if not next_link:
            break
        params["page"] = params["page"] + 1
    return {"data": all_items}


def show_and_save(name, df):
    """Print samenvatting, sla op als CSV en return de df."""
    if df.empty:
        print(f"[{name}] — geen data")
        return df
    path = os.path.join(CSV_DIR, f"{name}.csv")
    df.to_csv(path, index=False)
    print(f"[{name}] {len(df)} rijen, {len(df.columns)} kolommen → opgeslagen als {path}")
    return df

---
## Health Check

In [ ]:
raw = get("/api/public/health_check")
print("Ruwe response:", raw)
df_health = pd.DataFrame([raw]) if raw else pd.DataFrame()
show_and_save("health_check", df_health)
df_health

---
## Mail Templates

In [ ]:
raw = get("/api/public/mail_templates")
df_mail_templates = jsonapi_to_df(raw)
show_and_save("mail_templates", df_mail_templates)
df_mail_templates

---
## Processes

In [ ]:
raw = get("/api/public/processes")
df_processes = jsonapi_to_df(raw)
show_and_save("processes", df_processes)
df_processes

---
## Programs (alle pagina's)

In [ ]:
raw = fetch_paginated("/api/public/programs")
df_programs = jsonapi_to_df(raw)
show_and_save("programs", df_programs)
df_programs

---
## Tasks (alle pagina's)

In [ ]:
raw = fetch_paginated("/api/public/tasks")
df_tasks = jsonapi_to_df(raw)
show_and_save("tasks", df_tasks)
df_tasks

---
## Managed Organizations (alle pagina's)

In [ ]:
raw = fetch_paginated("/api/public/managed_organizations")
df_orgs = jsonapi_to_df(raw)
show_and_save("managed_organizations", df_orgs)
df_orgs

---
## Feedback – Candidate Feedbacks

In [ ]:
raw = get("/api/public/feedback/candidate_feedbacks")
df_feedbacks = jsonapi_to_df(raw)
show_and_save("feedback_candidate_feedbacks", df_feedbacks)
df_feedbacks

---
## Feedback – Templates

In [ ]:
raw = get("/api/public/feedback/templates")
df_feedback_templates = jsonapi_to_df(raw)
show_and_save("feedback_templates", df_feedback_templates)
df_feedback_templates

---
## Feedback – Respondent Groups

In [ ]:
raw = get("/api/public/feedback/groups")
df_groups = jsonapi_to_df(raw)
show_and_save("feedback_groups", df_groups)
df_groups

---
## Detail-endpoints (op basis van opgehaalde UUID's)
De cellen hieronder pakken automatisch de eerste UUID uit de lijsten hierboven en halen de details op.

### Process – detail

In [ ]:
if not df_processes.empty:
    uuid = df_processes.iloc[0]["id"]
    raw = get(f"/api/public/processes/{uuid}")
    df = jsonapi_to_df(raw)
    show_and_save("process_detail", df)
    display(df)
else:
    print("Geen processes beschikbaar.")

### Program – detail

In [ ]:
if not df_programs.empty:
    uuid = df_programs.iloc[0]["id"]
    raw = get(f"/api/public/programs/{uuid}")
    df = jsonapi_to_df(raw)
    show_and_save("program_detail", df)
    display(df)
else:
    print("Geen programs beschikbaar.")

### Task – detail

In [ ]:
if not df_tasks.empty:
    uuid = df_tasks.iloc[0]["id"]
    raw = get(f"/api/public/tasks/{uuid}")
    df = jsonapi_to_df(raw)
    show_and_save("task_detail", df)
    display(df)
else:
    print("Geen tasks beschikbaar.")

### Managed Organization – detail (incl. api-users)

In [ ]:
if not df_orgs.empty:
    uuid = df_orgs.iloc[0]["id"]
    raw = get(f"/api/public/managed_organizations/{uuid}")
    df = jsonapi_to_df(raw)
    show_and_save("managed_organization_detail", df)
    display(df)
    # api-users zitten in 'included'
    if raw and raw.get("included"):
        df_included = pd.json_normalize(raw["included"])
        show_and_save("managed_organization_api_users", df_included)
        display(df_included)
else:
    print("Geen managed organizations beschikbaar.")

### Feedback – candidate feedback detail

In [ ]:
if not df_feedbacks.empty:
    uuid = df_feedbacks.iloc[0]["id"]
    raw = get(f"/api/public/feedback/candidate_feedbacks/{uuid}")
    df = jsonapi_to_df(raw)
    show_and_save("feedback_candidate_feedback_detail", df)
    display(df)
else:
    print("Geen candidate feedbacks beschikbaar.")

### Feedback – template detail

In [ ]:
if not df_feedback_templates.empty:
    uuid = df_feedback_templates.iloc[0]["id"]
    raw = get(f"/api/public/feedback/templates/{uuid}")
    df = jsonapi_to_df(raw)
    show_and_save("feedback_template_detail", df)
    display(df)
else:
    print("Geen feedback templates beschikbaar.")

### Feedback – respondent group detail

In [ ]:
if not df_groups.empty:
    uuid = df_groups.iloc[0]["id"]
    raw = get(f"/api/public/feedback/groups/{uuid}")
    df = jsonapi_to_df(raw)
    show_and_save("feedback_group_detail", df)
    display(df)
else:
    print("Geen respondent groups beschikbaar.")

---
## Kandidaat-specifieke endpoints
Vul hier een bekende `candidate_uuid` in om details op te halen.
De UUID's van kandidaten zijn niet opvraagbaar via een lijst-endpoint.

In [ ]:
CANDIDATE_UUID = ""  # ← vul een UUID in

if CANDIDATE_UUID:
    # Kandidaat
    raw = get(f"/api/public/candidates/{CANDIDATE_UUID}")
    df = jsonapi_to_df(raw)
    show_and_save("candidate_detail", df)
    display(df)
else:
    print("Geen CANDIDATE_UUID ingevuld.")

In [ ]:
CANDIDATE_PROCESS_UUID = ""   # ← vul in
CANDIDATE_PROGRAM_UUID = ""   # ← vul in
CANDIDATE_TASK_UUID    = ""   # ← vul in
ASSIGNMENT_UUID        = ""   # ← vul in

for label, uuid, path_tpl in [
    ("candidate_process",       CANDIDATE_PROCESS_UUID, "/api/public/candidate_processes/{}"),
    ("candidate_program",       CANDIDATE_PROGRAM_UUID, "/api/public/candidate_programs/{}"),
    ("candidate_program_score", CANDIDATE_PROGRAM_UUID, "/api/public/candidate_programs/{}/score"),
    ("candidate_task",          CANDIDATE_TASK_UUID,    "/api/public/candidate_tasks/{}"),
    ("candidate_task_score",    CANDIDATE_TASK_UUID,    "/api/public/candidate_tasks/{}/score"),
    ("assignment",              ASSIGNMENT_UUID,        "/api/public/assignments/{}"),
]:
    if not uuid:
        print(f"[{label}] overgeslagen (geen UUID ingevuld)")
        continue
    raw = get(path_tpl.format(uuid))
    # score-endpoints returnen geen JSON:API envelope
    if "data" not in (raw or {}):
        df = pd.json_normalize(raw) if raw else pd.DataFrame()
    else:
        df = jsonapi_to_df(raw)
    show_and_save(label, df)
    display(df)

---
## Media Items (vereist user_id)

In [ ]:
USER_ID = ""  # ← vul een user UUID in

if USER_ID:
    raw = fetch_paginated(f"/api/public/users/{USER_ID}/media_items")
    df = jsonapi_to_df(raw)
    show_and_save("media_items_user", df)
    display(df)
else:
    print("Geen USER_ID ingevuld.")

---
## Overzicht opgeslagen CSV's

In [ ]:
files = sorted(os.listdir(CSV_DIR))
summary = []
for f in files:
    path = os.path.join(CSV_DIR, f)
    df = pd.read_csv(path)
    summary.append({"bestand": f, "rijen": len(df), "kolommen": len(df.columns)})
pd.DataFrame(summary)